# iGaming causal inference sandbox

This notebook creates a controlled campaign on top of session-derived features. Treatment assignment and individual effects are hidden from estimators but available for evaluation.

# План
- Посмотреть как выглядят данные
- Как он построил генерацию событий
- Как генерируется эффект
- Выделить в отдельные файлы разные алгоритмы 

In [2]:
from pathlib import Path
import sys
import pandas as pd

project = Path.cwd()
if not (project / 'sandbox.py').exists():
    project = project.parent
sys.path.insert(0, str(project))
from sandbox import prepare_session_units, simulate_campaign, evaluate


## 1. Derived analytical units

The raw source has no user ID. Each unit below represents one source session, split into pre-period covariates and a later natural outcome.

In [3]:
units = prepare_session_units()
print(f'{len(units)} usable session units')
units[['pre_event_count', 'pre_stake_eur', 'pre_ggr_eur', 'natural_future_ggr_eur']].describe().round(2)

369 usable session units


,pre_event_count,pre_stake_eur,pre_ggr_eur,natural_future_ggr_eur
count,369.00,369.00,369.00,369.00
mean,169.34,28.23,-0.78,1.69
std,196.71,62.64,32.05,33.51
min,4.00,0.20,-495.95,-243.78
25%,48.00,6.10,-1.60,-3.60
50%,103.00,15.15,1.88,3.20
75%,213.00,31.68,6.73,12.20
max,1492.00,1057.40,185.83,312.10


## 2. Four controlled worlds

`true_propensity_hidden` and `true_individual_effect_eur` exist only to validate the estimators; do not include them in a model.

In [8]:
simulate_campaign(units, '01_randomized', seed=27)

,unit_id,source_session_id,session_start,pre_event_count,pre_stake_eur,pre_ggr_eur,pre_deposit_eur,tenure_days,session_minutes,start_hour,month,is_weekend,natural_future_ggr_eur,country_synthetic,channel_synthetic,treatment_bonus,true_propensity_hidden,true_individual_effect_eur,future_ggr_eur,true_ate_eur
0,session_1,1,2024-02-01 10:38:19,68,36.05,25.06,50.0,0,16.116667,10,2,0,74.10,FI,direct,1,0.5,10.0,84.10,10.0
1,session_7,7,2024-03-02 15:23:56,32,20.00,-1.00,15.0,30,10.650000,15,3,1,16.00,GB,paid_search,0,0.5,10.0,16.00,10.0
2,session_9,9,2024-03-04 11:11:16,4,0.45,0.15,0.0,32,1.000000,11,3,0,-5.99,DE,organic,0,0.5,10.0,-5.99,10.0
3,session_10,10,2024-03-04 11:22:39,486,74.80,13.56,20.0,32,92.016667,11,3,0,17.25,CA,affiliate,0,0.5,10.0,17.25,10.0
4,session_12,12,2024-03-04 13:29:08,636,87.41,-92.49,0.0,32,193.350000,13,3,0,-65.03,GB,organic,1,0.5,10.0,-55.03,10.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
364,session_411,411,2025-10-18 21:19:36,166,28.04,10.95,20.0,625,51.433333,21,10,1,-6.11,GB,organic,0,0.5,10.0,-6.11,10.0
365,session_412,412,2025-10-20 20:33:17,280,44.50,-18.71,20.0,627,71.300000,20,10,0,38.30,FI,paid_search,0,0.5,10.0,38.30,10.0
366,session_415,415,2025-11-05 14:31:54,63,10.20,-12.94,20.0,643,20.750000,14,11,0,2.83,FI,direct,1,0.5,10.0,12.83,10.0
367,session_416,416,2025-11-07 12:28:42,153,30.60,5.11,20.0,645,38.316667,12,11,0,9.94,GB,organic,1,0.5,10.0,19.94,10.0


In [6]:
levels = ['01_randomized', '02_observable_confounding', '03_nonlinear_confounding', '04_heterogeneous_effect']
report = pd.concat([evaluate(simulate_campaign(units, level, seed=27), level) for level in levels], ignore_index=True)
report.sort_values(['level', 'absolute_error_eur']).round(3)

,level,method,estimate_ate_eur,true_ate_eur,bias_eur,absolute_error_eur,cate_rmse_eur
6,01_randomized,t_learner,10.310,10.000,0.310,0.310,21.723
8,01_randomized,x_learner,10.310,10.000,0.310,0.310,21.723
7,01_randomized,s_learner,10.310,10.000,0.310,0.310,21.723
0,01_randomized,naive,9.449,10.000,-0.551,0.551,0.551
5,01_randomized,dml,10.832,10.000,0.832,0.832,0.832
9,01_randomized,dr_learner,10.848,10.000,0.848,0.848,20.407
4,01_randomized,doubly_robust,10.848,10.000,0.848,0.848,0.848
3,01_randomized,ipw,10.926,10.000,0.926,0.926,0.926
1,01_randomized,regression_adjustment,11.637,10.000,1.637,1.637,1.637
2,01_randomized,propensity_score_matching,6.912,10.000,-3.088,3.088,3.088


In [7]:
report

,level,method,estimate_ate_eur,true_ate_eur,bias_eur,absolute_error_eur,cate_rmse_eur
0,01_randomized,naive,9.449387,10.00000,-0.550613,0.550613,0.550613
1,01_randomized,regression_adjustment,11.636708,10.00000,1.636708,1.636708,1.636708
2,01_randomized,propensity_score_matching,6.912330,10.00000,-3.087670,3.087670,3.087670
3,01_randomized,ipw,10.926496,10.00000,0.926496,0.926496,0.926496
4,01_randomized,doubly_robust,10.847628,10.00000,0.847628,0.847628,0.847628
5,01_randomized,dml,10.832101,10.00000,0.832101,0.832101,0.832101
6,01_randomized,t_learner,10.309599,10.00000,0.309599,0.309599,21.722569
7,01_randomized,s_learner,10.309599,10.00000,0.309599,0.309599,21.722569
8,01_randomized,x_learner,10.309599,10.00000,0.309599,0.309599,21.722569
9,01_randomized,dr_learner,10.847628,10.00000,0.847628,0.847628,20.406963


## 3. Heterogeneous-effect evaluation

CATE RMSE is the relevant metric for targeting: it measures whether the model recovers *who* should receive a bonus, not just the average effect.

In [ ]:
heterogeneous = report.query("level == '04_heterogeneous_effect'").sort_values('cate_rmse_eur')
heterogeneous[['method', 'estimate_ate_eur', 'true_ate_eur', 'absolute_error_eur', 'cate_rmse_eur']].round(3)